# Summary

Build a vertex AI Search App

In [1]:
import os, sys
import pandas as pd
import json
#
# from typing import List
#
# # GCP
# from google.api_core.client_options import ClientOptions
# from google.cloud import discoveryengine_v1 as discoveryengine
# from google.protobuf import field_mask_pb2

import vai_search_app_build as vsab

# Numantic utilities
utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")


## Create a Vertex AI data store

In [2]:
project_id = os.environ["GOOGLE_CLOUD_PROJECT_ID"]
location = "global"
data_store_id = "rag-tests-datastore-v26"
display_name = "rag-tests-datastore-v26"
# bigquery_dataset = "ns_bq"
# bigquery_table = "rag_tests_3"


In [3]:
ds_name = vsab.create_vais_data_store(project_id=project_id,
                                      location=location,
                                      data_store_id=data_store_id,
                                      display_name=display_name
                                      )
print(ds_name)

Creating Data Store: rag-tests-datastore-v26...
projects/643773332888/locations/global/collections/default_collection/dataStores/rag-tests-datastore-v26


## Update data store schema

In [6]:
# Define the data store's schema
schema_dict = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "keyPropertyMapping": "title",
            "retrievable": True
        },
        "source_url": {
            "type": "string",
            "keyPropertyMapping": "uri",
            "retrievable": True
        },
        "source_type": {
            "type": "string",
            "retrievable": True,
            "indexable": True,
            "dynamicFacetable": True  # Allows you to filter by 'gaming', 'recipes', etc.
        },
        "doc_index": {
            "type": "string",
            "retrievable": True,
            "indexable": True
        }
    }
}


In [7]:
vsab.update_vais_schema(project_id=project_id,
                        location=location,
                        data_store_id=data_store_id,
                        schema_dict=schema_dict
                        )


Updating schema for rag-tests-datastore-v26...
Schema updated successfully: projects/643773332888/locations/global/collections/default_collection/dataStores/rag-tests-datastore-v26/schemas/default_schema


## Load documents into the data store

In [3]:
source_uri = "gs://ns_datasets/rag-tests/documents/metadata.jsonl"

In [4]:
vsab.import_documents_to_data_store(project_id=project_id,
                                    location=location,
                                    data_store_id=data_store_id,
                                    source_uri=source_uri
                                    )


Importing data from GCS source...


'projects/643773332888/locations/global/collections/default_collection/dataStores/rag-tests-datastore-v26/branches/0/operations/import-documents-13810436240252652194'

## Check document count

In [5]:
vsab.get_document_count(project_id=project_id,
                        location=location,
                        data_store_id=data_store_id
                        )


Data Store 'rag-tests-datastore-v26' contains 20 documents.


20

## Create a Vertex AI search app (engine)

In [6]:
app_engine_id = "rag-tests-searchapp-v26"
app_display_name = "rag-tests-searchapp-v26"

In [7]:
vsab.create_vai_search_engine(project_id=project_id,
                              location=location,
                              engine_id=app_engine_id,
                              display_name=app_display_name,
                              data_store_ids=[data_store_id]
                              )


Creating search engine: rag-tests-searchapp-v26...


'projects/643773332888/locations/global/collections/default_collection/engines/rag-tests-searchapp-v26'

## Create a Vertex AI Data Store

In [3]:
def create_vais_data_store(project_id: str, location: str, data_store_id: str, display_name: str) -> str:
    client = discoveryengine.DataStoreServiceClient()
    parent = f"projects/{project_id}/locations/{location}/collections/default_collection"

    data_store = discoveryengine.DataStore(
        display_name=display_name,
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        # CHANGE: Use CONTENT_OPTIONAL to allow BigQuery-based text
        content_config=discoveryengine.DataStore.ContentConfig.NO_CONTENT,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
    )

    operation = client.create_data_store(parent=parent,
                                         data_store=data_store,
                                         data_store_id=data_store_id)
    print(f"Creating Data Store: {data_store_id}...")
    response = operation.result()
    return response.name

In [6]:
project_id = os.environ["GOOGLE_CLOUD_PROJECT_ID"]
location = "global"
data_store_id = "rag-tests-datastore-v17"
display_name = "rag-tests-datastore-v17"
bigquery_dataset = "ns_bq"
bigquery_table = "rag_tests_3"


In [7]:
# 1. Create the Data Store
data_store_name = create_vais_data_store(project_id=project_id,
                                         location=location,
                                         data_store_id=data_store_id,
                                         display_name=display_name)


Creating Data Store: rag-tests-datastore-v17...


## Update schema

In [8]:

def update_schema_for_rag(project_id: str, location: str, data_store_id: str):
    client = discoveryengine.SchemaServiceClient()
    schema_name = client.schema_path(project_id, location, data_store_id, "default_schema")

    # Define your schema exactly as before
    schema_dict = {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "type": "object",
        "properties": {
            "_id": {
                "type": "string",
                "retrievable": True
                # No mapping here; system handles the internal ID automatically
            },
            "title": {
                "type": "string",
                "keyPropertyMapping": "title",
                "retrievable": True,
                "completable": True
                # Removed searchable/indexable (implied by 'title' mapping)
            },
            "content": {
                "type": "string",
                "keyPropertyMapping": "description",
                "retrievable": True
                # Removed searchable/indexable (implied by 'description' mapping)
            },
            "source_url": {
                "type": "string",
                "keyPropertyMapping": "uri",
                "retrievable": True
                # Removed searchable/indexable (implied by 'uri' mapping)
            },
            "source_type": {
                "type": "string",
                "retrievable": True,
                "indexable": True,
                "dynamicFacetable": True
                # Mapping is blank, so explicit annotations ARE allowed here
            },
            "doc_index": {
                "type": "string",
                "retrievable": True,
                "indexable": True,
                "dynamicFacetable": True,
                "searchable": True
                # Mapping is blank, so explicit annotations ARE allowed here
            }
        }
    }

    schema = discoveryengine.Schema(
        name=schema_name,
        json_schema=json.dumps(schema_dict)
    )
    schema = discoveryengine.Schema(
        name=schema_name,
        json_schema=json.dumps(schema_dict)
    )

    # Simplified Request: Discovery Engine Schema updates do not use update_mask
    request = discoveryengine.UpdateSchemaRequest(schema=schema)

    print(f"Updating schema for {data_store_id}...")
    try:
        operation = client.update_schema(request=request)
        response = operation.result()
        print(f"Schema updated successfully: {response.name}")
    except Exception as e:
        print(f"Error updating schema: {e}")

In [9]:
update_schema_for_rag(project_id=project_id,
                      location=location,
                      data_store_id=data_store_id)


Updating schema for rag-tests-datastore-v17...
Schema updated successfully: projects/643773332888/locations/global/collections/default_collection/dataStores/rag-tests-datastore-v17/schemas/default_schema


## Ingest documents into the data store

In [10]:
def import_documents_from_bigquery(project_id: str,
                                   location: str,
                                   data_store_id: str,
                                   bigquery_dataset: str,
                                   bigquery_table: str):
    client = discoveryengine.DocumentServiceClient()

    parent = client.branch_path(project_id, location, data_store_id, "default_branch")

    bigquery_source = discoveryengine.BigQuerySource(
        project_id=project_id,
        dataset_id=bigquery_dataset,
        table_id=bigquery_table,
        # 'custom' is standard for non-structured tables where you want to map fields like 'content'
        data_schema="custom"
    )

    request = discoveryengine.ImportDocumentsRequest(
        parent=parent,
        bigquery_source=bigquery_source,
        reconciliation_mode=discoveryengine.ImportDocumentsRequest.ReconciliationMode.INCREMENTAL,
    )

    print(f"Importing data from {bigquery_table}...")
    operation = client.import_documents(request=request)
    return operation.operation.name


In [11]:
# 2. Ingest data from BigQuery
import_documents_from_bigquery(project_id=project_id,
                               location=location,
                               data_store_id=data_store_id,
                               bigquery_dataset=bigquery_dataset,
                               bigquery_table=bigquery_table)


Importing data from rag_tests_3...


'projects/643773332888/locations/global/collections/default_collection/dataStores/rag-tests-datastore-v17/branches/0/operations/import-documents-12019816683196887716'

In [12]:
def get_real_document_count(project_id: str,
                            location: str,
                            data_store_id: str) -> int:
    """
    Counts total documents in the default branch of a data store.
    """
    client_options = ClientOptions(
        api_endpoint=f"{location}-discoveryengine.googleapis.com" if location != "global" else None
    )
    client = discoveryengine.DocumentServiceClient(client_options=client_options)

    # Path to the default branch (where BigQuery imports land)
    parent = client.branch_path(
        project=project_id,
        location=location,
        data_store=data_store_id,
        branch="default_branch"
    )

    # Use ListDocuments with a small page size just to get the total size
    # In version 0.13.12, the list_documents method returns a pager
    # that you can iterate or check for total size.
    request = discoveryengine.ListDocumentsRequest(
        parent=parent,
        page_size=1000 # Max allowed to minimize API calls
    )

    results = client.list_documents(request=request)

    # Iterate through the pager to get the total count
    total_count = sum(1 for _ in results)

    print(f"Total documents found in '{data_store_id}': {total_count}")
    return total_count

In [13]:
get_real_document_count(project_id=project_id,
                        location=location,
                        data_store_id=data_store_id)


Total documents found in 'rag-tests-datastore-v17': 1391


1391

## Check schema

In [14]:
def get_vais_schema(project_id: str,
                    location: str,
                    data_store_id: str):
    """
    Retrieves the current schema for a specific data store.
    """
    client = discoveryengine.SchemaServiceClient()

    # The resource name for the default schema
    schema_name = client.schema_path(
        project_id,
        location,
        data_store_id,
        "default_schema"
    )

    try:
        print(f"Fetching schema: {schema_name}...")
        response = client.get_schema(name=schema_name)

        # The actual JSON structure is stored as a string in the 'json_schema' field
        current_schema = json.loads(response.json_schema)

        print("\n--- Current Data Store Schema ---")
        print(json.dumps(current_schema, indent=2))

        return current_schema
    except Exception as e:
        print(f"Error retrieving schema: {e}")
        return None



In [15]:
# Usage
get_vais_schema(project_id=project_id,
                location=location,
                data_store_id=data_store_id)


Fetching schema: projects/ns-research-q4-2025/locations/global/dataStores/rag-tests-datastore-v17/schemas/default_schema...

--- Current Data Store Schema ---
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "properties": {
    "_id": {
      "retrievable": true,
      "type": "string"
    },
    "title": {
      "type": "string",
      "retrievable": true,
      "completable": true,
      "keyPropertyMapping": "title"
    },
    "doc_index": {
      "dynamicFacetable": true,
      "indexable": true,
      "retrievable": true,
      "type": "string",
      "searchable": true
    },
    "source_url": {
      "keyPropertyMapping": "uri",
      "type": "string",
      "retrievable": true
    },
    "source_type": {
      "type": "string",
      "retrievable": true,
      "indexable": true,
      "dynamicFacetable": true
    },
    "content": {
      "keyPropertyMapping": "description",
      "retrievable": true,
      "type": "string"
    }
  }
}


{'$schema': 'https://json-schema.org/draft/2020-12/schema',
 'type': 'object',
 'properties': {'_id': {'retrievable': True, 'type': 'string'},
  'title': {'type': 'string',
   'retrievable': True,
   'completable': True,
   'keyPropertyMapping': 'title'},
  'doc_index': {'dynamicFacetable': True,
   'indexable': True,
   'retrievable': True,
   'type': 'string',
   'searchable': True},
  'source_url': {'keyPropertyMapping': 'uri',
   'type': 'string',
   'retrievable': True},
  'source_type': {'type': 'string',
   'retrievable': True,
   'indexable': True,
   'dynamicFacetable': True},
  'content': {'keyPropertyMapping': 'description',
   'retrievable': True,
   'type': 'string'}}}

## Create a Vertex AI Search App

In [16]:
def create_search_engine_from_datastore(project_id: str,
                                        location: str,
                                        engine_id: str,
                                        display_name: str,
                                        data_store_ids: list):
    client = discoveryengine.EngineServiceClient()

    parent = f"projects/{project_id}/locations/{location}/collections/default_collection"

    engine = discoveryengine.Engine(
        display_name=display_name,
        solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        data_store_ids=data_store_ids,
        search_engine_config=discoveryengine.Engine.SearchEngineConfig(
            # Enterprise tier is required for LLM/RAG features
            search_tier=discoveryengine.SearchTier.SEARCH_TIER_ENTERPRISE,
            search_add_ons=[discoveryengine.SearchAddOn.SEARCH_ADD_ON_LLM]
        )
    )

    operation = client.create_engine(
        parent=parent,
        engine=engine,
        engine_id=engine_id
    )

    print(f"Creating engine {display_name}...")
    response = operation.result()
    return response.name



In [17]:
app_engine_id = "rag-tests-searchapp-v17"
app_display_name = "rag-tests-searchapp-v17"

create_search_engine_from_datastore(project_id=project_id,
                                    location=location,
                                    engine_id=app_engine_id,
                                    display_name=app_display_name,
                                    data_store_ids=[data_store_id]
                                    )


Creating engine rag-tests-searchapp-v17...


'projects/643773332888/locations/global/collections/default_collection/engines/rag-tests-searchapp-v17'